# LangChain Deep Calculator Agent

LangChain Deep Agents 기반 계산기 Agent 예제입니다.

구성 Tool:
- `calculate_expression`: 일반 수식 계산
- `calculate_percentage`: 백분율 계산
- `calculate_statistics`: 기본 통계 계산
- `convert_unit`: 단위 변환


## 1. 패키지 설치

Google Colab 또는 신규 Jupyter 환경에서 실행합니다.

In [ ]:
%pip install -U deepagents langchain langchain-openai

## 2. OpenAI API Key 설정

Colab에서는 입력값이 화면에 노출되지 않도록 `getpass`를 사용합니다.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

print("OPENAI_API_KEY 설정 완료")

## 3. 공통 모듈 및 안전한 수식 계산기 정의

In [ ]:
from __future__ import annotations

import ast
import math
import operator
import statistics
from typing import Literal

from deepagents import create_deep_agent
from langchain.tools import tool


ALLOWED_BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}

ALLOWED_UNARY_OPERATORS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt,
    "abs": abs,
    "round": round,
    "ceil": math.ceil,
    "floor": math.floor,
    "sin": math.sin,
    "cos": math.cos,
    "tan": math.tan,
    "log": math.log,
    "log10": math.log10,
    "exp": math.exp,
}

ALLOWED_CONSTANTS = {
    "pi": math.pi,
    "e": math.e,
}


def _validate_numeric_result(value: int | float) -> int | float:
    if isinstance(value, complex):
        raise ValueError("Complex-number results are not supported.")
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("The calculation produced a non-finite result.")
    return value


def _safe_evaluate_node(node: ast.AST) -> int | float:
    if isinstance(node, ast.Expression):
        return _safe_evaluate_node(node.body)

    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool):
            raise ValueError("Boolean values are not supported.")
        if isinstance(node.value, (int, float)):
            return _validate_numeric_result(node.value)
        raise ValueError("Only numeric constants are allowed.")

    if isinstance(node, ast.BinOp):
        operator_function = ALLOWED_BINARY_OPERATORS.get(type(node.op))
        if operator_function is None:
            raise ValueError(f"Unsupported binary operator: {type(node.op).__name__}")

        left_value = _safe_evaluate_node(node.left)
        right_value = _safe_evaluate_node(node.right)

        if isinstance(node.op, (ast.Div, ast.FloorDiv, ast.Mod)) and right_value == 0:
            raise ZeroDivisionError("Division by zero is not allowed.")

        if isinstance(node.op, ast.Pow) and abs(right_value) > 100:
            raise ValueError("The exponent is too large. Use an absolute exponent value of 100 or less.")

        return _validate_numeric_result(operator_function(left_value, right_value))

    if isinstance(node, ast.UnaryOp):
        operator_function = ALLOWED_UNARY_OPERATORS.get(type(node.op))
        if operator_function is None:
            raise ValueError(f"Unsupported unary operator: {type(node.op).__name__}")
        return _validate_numeric_result(operator_function(_safe_evaluate_node(node.operand)))

    if isinstance(node, ast.Name):
        if node.id not in ALLOWED_CONSTANTS:
            raise ValueError(f"Unsupported name: {node.id}")
        return ALLOWED_CONSTANTS[node.id]

    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name):
            raise ValueError("Only explicitly supported functions may be called.")

        function_name = node.func.id
        function = ALLOWED_FUNCTIONS.get(function_name)
        if function is None:
            raise ValueError(f"Unsupported function: {function_name}")
        if node.keywords:
            raise ValueError("Keyword arguments are not supported.")

        arguments = [_safe_evaluate_node(argument) for argument in node.args]
        try:
            result = function(*arguments)
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError(f"Invalid arguments for {function_name}: {error}") from error

        return _validate_numeric_result(result)

    raise ValueError(f"Unsupported expression element: {type(node).__name__}")


def safe_evaluate(expression: str) -> int | float:
    if not expression or not expression.strip():
        raise ValueError("The expression cannot be empty.")
    if len(expression) > 500:
        raise ValueError("The expression is too long.")

    normalized_expression = (
        expression.strip()
        .replace("^", "**")
        .replace("×", "*")
        .replace("÷", "/")
    )

    try:
        parsed_expression = ast.parse(normalized_expression, mode="eval")
    except SyntaxError as error:
        raise ValueError(f"Invalid mathematical expression: {error.msg}") from error

    return _safe_evaluate_node(parsed_expression)


def _format_number(value: int | float, precision: int | None) -> int | float:
    if precision is None:
        return value
    if precision < 0 or precision > 15:
        raise ValueError("Precision must be between 0 and 15.")
    return round(value, precision)

## 4. 계산 Tool 정의

In [ ]:
@tool
def calculate_expression(expression: str, precision: int | None = None) -> dict:
    """Safely evaluate arithmetic, powers, roots, logarithms and supported math functions."""
    try:
        normalized_expression = (
            expression.strip().replace("^", "**").replace("×", "*").replace("÷", "/")
        )
        result = safe_evaluate(expression)
        return {
            "success": True,
            "normalized_expression": normalized_expression,
            "result": _format_number(result, precision),
            "precision": precision,
            "error": None,
        }
    except Exception as error:
        return {
            "success": False,
            "normalized_expression": expression,
            "result": None,
            "precision": precision,
            "error": str(error),
        }


PercentageOperation = Literal[
    "percentage_of", "percentage_change", "increase_by_percentage",
    "decrease_by_percentage", "percentage_point_difference",
    "reverse_percentage", "discount", "add_tax"
]


@tool
def calculate_percentage(
    operation: PercentageOperation,
    base_value: float | None = None,
    percentage: float | None = None,
    new_value: float | None = None,
    original_value: float | None = None,
    precision: int | None = None,
) -> dict:
    """Calculate percentages, percentage changes, discounts, taxes and percentage-point differences."""
    try:
        if operation == "percentage_of":
            if base_value is None or percentage is None:
                raise ValueError("percentage_of requires base_value and percentage.")
            result = base_value * percentage / 100
            formula = f"{base_value} × ({percentage} / 100)"
        elif operation == "percentage_change":
            if original_value is None or new_value is None:
                raise ValueError("percentage_change requires original_value and new_value.")
            if original_value == 0:
                raise ZeroDivisionError("Percentage change cannot use an original value of zero.")
            result = (new_value - original_value) / abs(original_value) * 100
            formula = f"(({new_value} - {original_value}) / |{original_value}|) × 100"
        elif operation == "increase_by_percentage":
            if base_value is None or percentage is None:
                raise ValueError("increase_by_percentage requires base_value and percentage.")
            result = base_value * (1 + percentage / 100)
            formula = f"{base_value} × (1 + {percentage} / 100)"
        elif operation == "decrease_by_percentage":
            if base_value is None or percentage is None:
                raise ValueError("decrease_by_percentage requires base_value and percentage.")
            result = base_value * (1 - percentage / 100)
            formula = f"{base_value} × (1 - {percentage} / 100)"
        elif operation == "percentage_point_difference":
            if original_value is None or new_value is None:
                raise ValueError("percentage_point_difference requires original_value and new_value.")
            result = new_value - original_value
            formula = f"{new_value}% - {original_value}%"
        elif operation == "reverse_percentage":
            if base_value is None or percentage is None:
                raise ValueError("reverse_percentage requires base_value and percentage.")
            multiplier = 1 + percentage / 100
            if multiplier == 0:
                raise ZeroDivisionError("The reverse-percentage multiplier cannot be zero.")
            result = base_value / multiplier
            formula = f"{base_value} / (1 + {percentage} / 100)"
        elif operation == "discount":
            if base_value is None or percentage is None:
                raise ValueError("discount requires base_value and percentage.")
            if percentage < 0 or percentage > 100:
                raise ValueError("A discount percentage must be between 0 and 100.")
            result = base_value * (1 - percentage / 100)
            formula = f"{base_value} × (1 - {percentage} / 100)"
        elif operation == "add_tax":
            if base_value is None or percentage is None:
                raise ValueError("add_tax requires base_value and percentage.")
            result = base_value * (1 + percentage / 100)
            formula = f"{base_value} × (1 + {percentage} / 100)"
        else:
            raise ValueError(f"Unsupported operation: {operation}")

        return {
            "success": True,
            "operation": operation,
            "formula": formula,
            "result": _format_number(result, precision),
            "precision": precision,
            "error": None,
        }
    except Exception as error:
        return {
            "success": False,
            "operation": operation,
            "formula": None,
            "result": None,
            "precision": precision,
            "error": str(error),
        }


StatisticsOperation = Literal[
    "sum", "count", "mean", "median", "minimum", "maximum",
    "range", "variance", "standard_deviation"
]
VarianceType = Literal["population", "sample"]


@tool
def calculate_statistics(
    values: list[float],
    operations: list[StatisticsOperation],
    variance_type: VarianceType = "population",
    precision: int | None = None,
) -> dict:
    """Calculate descriptive statistics for a numeric list."""
    try:
        if not values:
            raise ValueError("At least one numeric value is required.")
        if not operations:
            raise ValueError("At least one statistical operation is required.")
        if any(not math.isfinite(value) for value in values):
            raise ValueError("All input values must be finite numbers.")

        results = {}
        for operation_name in operations:
            if operation_name == "sum": result = sum(values)
            elif operation_name == "count": result = len(values)
            elif operation_name == "mean": result = statistics.fmean(values)
            elif operation_name == "median": result = statistics.median(values)
            elif operation_name == "minimum": result = min(values)
            elif operation_name == "maximum": result = max(values)
            elif operation_name == "range": result = max(values) - min(values)
            elif operation_name == "variance":
                if variance_type == "population": result = statistics.pvariance(values)
                else:
                    if len(values) < 2: raise ValueError("Sample variance requires at least two values.")
                    result = statistics.variance(values)
            elif operation_name == "standard_deviation":
                if variance_type == "population": result = statistics.pstdev(values)
                else:
                    if len(values) < 2: raise ValueError("Sample standard deviation requires at least two values.")
                    result = statistics.stdev(values)
            else:
                raise ValueError(f"Unsupported statistics operation: {operation_name}")

            results[operation_name] = _format_number(result, precision) if isinstance(result, float) else result

        return {
            "success": True,
            "value_count": len(values),
            "variance_type": variance_type,
            "results": results,
            "precision": precision,
            "error": None,
        }
    except Exception as error:
        return {
            "success": False,
            "value_count": len(values) if values else 0,
            "variance_type": variance_type,
            "results": {},
            "precision": precision,
            "error": str(error),
        }

In [ ]:
UnitDimension = Literal[
    "length", "area", "volume", "mass", "time", "speed",
    "temperature", "pressure", "energy", "power", "data"
]

UNIT_ALIASES = {
    "meter": "m", "meters": "m", "kilometer": "km", "kilometers": "km",
    "centimeter": "cm", "centimeters": "cm", "millimeter": "mm", "millimeters": "mm",
    "inch": "in", "inches": "in", "foot": "ft", "feet": "ft",
    "yard": "yd", "yards": "yd", "mile": "mi", "miles": "mi",
    "kilogram": "kg", "kilograms": "kg", "gram": "g", "grams": "g",
    "milligram": "mg", "milligrams": "mg", "pound": "lb", "pounds": "lb",
    "ounce": "oz", "ounces": "oz", "second": "s", "seconds": "s",
    "minute": "min", "minutes": "min", "hour": "h", "hours": "h",
    "day": "day", "days": "day", "celsius": "c", "fahrenheit": "f",
    "kelvin": "k", "byte": "b", "bytes": "b", "kilobyte": "kb",
    "kilobytes": "kb", "megabyte": "mb", "megabytes": "mb",
    "gigabyte": "gb", "gigabytes": "gb", "terabyte": "tb", "terabytes": "tb",
}

LINEAR_UNIT_FACTORS = {
    "length": {"mm": 0.001, "cm": 0.01, "m": 1.0, "km": 1000.0, "in": 0.0254, "ft": 0.3048, "yd": 0.9144, "mi": 1609.344},
    "area": {"mm2": 0.000001, "cm2": 0.0001, "m2": 1.0, "km2": 1_000_000.0, "in2": 0.00064516, "ft2": 0.09290304, "acre": 4046.8564224, "ha": 10_000.0},
    "volume": {"ml": 0.000001, "l": 0.001, "m3": 1.0, "cm3": 0.000001, "in3": 0.000016387064, "ft3": 0.028316846592, "gal_us": 0.003785411784},
    "mass": {"mg": 0.000001, "g": 0.001, "kg": 1.0, "t": 1000.0, "oz": 0.028349523125, "lb": 0.45359237},
    "time": {"ms": 0.001, "s": 1.0, "min": 60.0, "h": 3600.0, "day": 86400.0},
    "speed": {"m/s": 1.0, "km/h": 0.2777777777777778, "mph": 0.44704, "ft/s": 0.3048, "knot": 0.5144444444444445},
    "pressure": {"pa": 1.0, "kpa": 1000.0, "mpa": 1_000_000.0, "bar": 100_000.0, "atm": 101_325.0, "psi": 6894.757293168},
    "energy": {"j": 1.0, "kj": 1000.0, "mj": 1_000_000.0, "wh": 3600.0, "kwh": 3_600_000.0, "cal": 4.184, "kcal": 4184.0},
    "power": {"w": 1.0, "kw": 1000.0, "mw": 1_000_000.0, "hp": 745.6998715822702},
    "data": {"b": 1.0, "kb": 1024.0, "mb": 1024.0**2, "gb": 1024.0**3, "tb": 1024.0**4},
}


def _normalize_unit(unit: str) -> str:
    normalized = unit.strip().lower().replace("²", "2").replace("³", "3")
    return UNIT_ALIASES.get(normalized, normalized)


def _convert_temperature(value: float, from_unit: str, to_unit: str) -> tuple[float, str]:
    supported_units = {"c", "f", "k"}
    if from_unit not in supported_units or to_unit not in supported_units:
        raise ValueError("Temperature units must be c, f or k.")

    if from_unit == "c": celsius = value
    elif from_unit == "f": celsius = (value - 32) * 5 / 9
    else: celsius = value - 273.15

    if celsius < -273.15:
        raise ValueError("The input temperature is below absolute zero.")

    if to_unit == "c":
        return celsius, "Convert source to Celsius; return Celsius."
    if to_unit == "f":
        return celsius * 9 / 5 + 32, "Convert source to Celsius; °F = °C × 9/5 + 32."
    return celsius + 273.15, "Convert source to Celsius; K = °C + 273.15."


@tool
def convert_unit(
    value: float,
    from_unit: str,
    to_unit: str,
    dimension: UnitDimension,
    precision: int | None = None,
) -> dict:
    """Convert a numeric value between compatible physical units."""
    try:
        if not math.isfinite(value):
            raise ValueError("The input value must be a finite number.")

        normalized_from_unit = _normalize_unit(from_unit)
        normalized_to_unit = _normalize_unit(to_unit)

        if dimension == "temperature":
            converted_value, formula = _convert_temperature(value, normalized_from_unit, normalized_to_unit)
        else:
            dimension_units = LINEAR_UNIT_FACTORS.get(dimension)
            if dimension_units is None:
                raise ValueError(f"Unsupported unit dimension: {dimension}")
            if normalized_from_unit not in dimension_units:
                raise ValueError(f"Unsupported source unit '{from_unit}' for dimension '{dimension}'.")
            if normalized_to_unit not in dimension_units:
                raise ValueError(f"Unsupported target unit '{to_unit}' for dimension '{dimension}'.")

            source_factor = dimension_units[normalized_from_unit]
            target_factor = dimension_units[normalized_to_unit]
            converted_value = value * source_factor / target_factor
            formula = f"{value} × {source_factor} ÷ {target_factor}"

        return {
            "success": True,
            "dimension": dimension,
            "input_value": value,
            "from_unit": normalized_from_unit,
            "to_unit": normalized_to_unit,
            "formula": formula,
            "result": _format_number(converted_value, precision),
            "precision": precision,
            "error": None,
        }
    except Exception as error:
        return {
            "success": False,
            "dimension": dimension,
            "input_value": value,
            "from_unit": from_unit,
            "to_unit": to_unit,
            "formula": None,
            "result": None,
            "precision": precision,
            "error": str(error),
        }

## 5. System Prompt 정의

In [ ]:
CALCULATOR_SYSTEM_PROMPT = """
You are a reliable calculator agent.

Your responsibility is to interpret the user's calculation request,
select the correct calculator tool, validate inputs, and return an
accurate and clearly formatted result.

Available tools:
1. calculate_expression: arithmetic, powers, roots, logarithms and math functions.
2. calculate_percentage: percentages, percentage change, discounts, taxes and percentage points.
3. calculate_statistics: sum, mean, median, range, variance and standard deviation.
4. convert_unit: compatible unit conversions.

Mandatory rules:
- Use a calculator tool for every numeric calculation.
- Do not rely on mental arithmetic when a suitable tool is available.
- Do not invent missing values.
- Preserve standard operator precedence and units.
- Detect division by zero and invalid mathematical domains.
- Distinguish percentage change from percentage-point change.
- Distinguish sample statistics from population statistics.
- Never fabricate a result when a tool reports an error.
- Round only when requested or needed for presentation.
- Respond in Korean unless the user requests another language.

Simple response format:
결과: <value and unit>

Detailed response format:
계산식 또는 해석: <expression>
결과: <value and unit>
설명: <brief explanation>
"""

## 6. Deep Calculator Agent 생성

`openai:gpt-5.5`가 계정 또는 실행 환경에서 지원되지 않으면 사용 가능한 모델 식별자로 변경합니다.

In [ ]:
MODEL_NAME = "openai:gpt-5.5"

calculator_agent = create_deep_agent(
    model=MODEL_NAME,
    tools=[
        calculate_expression,
        calculate_percentage,
        calculate_statistics,
        convert_unit,
    ],
    system_prompt=CALCULATOR_SYSTEM_PROMPT,
)

print("Calculator Agent 생성 완료")

## 7. Agent 호출 함수

In [ ]:
def ask_calculator(question: str) -> str:
    if not question or not question.strip():
        raise ValueError("질문을 입력해 주세요.")

    result = calculator_agent.invoke({
        "messages": [{"role": "user", "content": question.strip()}]
    })

    content = result["messages"][-1].content
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []
        for block in content:
            if isinstance(block, str):
                text_parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                text_parts.append(str(block.get("text", "")))
        return "\n".join(text_parts).strip()

    return str(content)

## 8. 실행 테스트

In [ ]:
print(ask_calculator("25 곱하기 18은?"))

In [ ]:
print(ask_calculator("12만 원에서 15% 할인한 가격을 계산해 주세요."))

In [ ]:
print(ask_calculator("10, 20, 30, 40, 50의 평균과 모집단 표준편차를 계산해 주세요."))

In [ ]:
print(ask_calculator("시속 90km를 초속 m/s로 변환해 주세요."))

## 9. 대화형 입력

아래 셀을 반복 실행하면서 다른 계산 요청을 테스트할 수 있습니다.

In [ ]:
question = input("계산 요청: ")
print(ask_calculator(question))